# Evaluación de la eficacia de controles y conclusión de oportunidades de mejora

In [1]:
from utils import exportar_csv, calcular_residual, etiqueta_prob, etiqueta_imp, zona_riesgo
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

### Cargar la data que tiene los seguimiento, el reporte de ISOLUCION

In [2]:
# Fuentes originales - isolucion
mapa_iso = pd.read_excel('Reporte Mejoramiento Continuo - Controles.xlsx', 
                        sheet_name='Data', header=None)

### Crear la data de seguimientos

In [3]:
# Generar un df únicamente con los seguimientos que se registraron en ISOLUCION por control
seguimientos = mapa_iso[[1, 13, 17, 18, 16]].copy()
# Generar nombre de las columnas para la data de seguimientos
seguimientos.columns = ['num', 'actividad', 'seguimiento', 'fecha_seguimiento', 'eficacia']
# Excluir las filas de encabezado que ya no se requieren
seguimientos = seguimientos.iloc[2:].reset_index(drop=True)
# Rellenar números
seguimientos['num'] = seguimientos['num'].ffill()

In [4]:
# Identificar la fila que tiene el texto del control
seguimientos['es_cabecera'] = seguimientos['actividad'].notna()
# Contar la cantidad de controles (cabecera) por número de riesgo num
seguimientos['no_control'] = seguimientos.groupby('num')['es_cabecera'].cumsum()
# Crear la llave del riesgo uniendo num y el consecutivo
seguimientos['id_control_iso'] = (seguimientos['num'].astype(int).astype(str) + 
                                   '-' + seguimientos['no_control'].astype(str))

In [5]:
# Contar el tamaño en caracteres de cada seguimiento
seguimientos['len_seguimiento'] = seguimientos['seguimiento'].astype(str).apply(len)

In [6]:
# Eliminar controles duplicados confirmados en Isolucion
seguimientos = seguimientos[seguimientos['id_control_iso'] != '5431-2'].reset_index(drop=True)

### Identificar las oportunidades de mejora

#### Filtrar solo los últimos seguimientos y seleccionar los números asignados

In [7]:
# Tomar solo el último seguimiento de cada control, es decir, el de la fecha más reciente
ultimos = seguimientos.sort_values('fecha_seguimiento').groupby(
    'id_control_iso', as_index=False
).last()

In [8]:
# Planes de controles de gestión evaluados por JASD
nums_evaluados = [
    5440, 5441, 5453, 5459, 5460, 5461, 5463, 5464, 5465, 5466,
    5467, 5468, 5469, 5470, 5471, 5472, 5473, 5474, 5476,
    5542, 5543, 5544, 5546, 5549, 5551, 5552, 5555, 5560
]

In [9]:
# De los últimos seguimientos tomar solo los asignados
# Esto aplica en este caso porque solo se están tomando los asignados a JASD
ultimos['num'] = ultimos['id_control_iso'].str.split('-').str[0].astype(int)
evaluaciones = ultimos[ultimos['num'].isin(nums_evaluados)].copy()
evaluaciones = evaluaciones[['id_control_iso','num','seguimiento','eficacia']]

print(f"Total: {len(evaluaciones)}")

Total: 52


In [10]:
ef_global = pd.read_csv('comentarios_eficacia_global.csv', encoding='utf-8-sig')

## Extraer oportunidades de mejora

#### Definición de categorías de problemas

In [11]:
# Crear un estándar de patrones de problemas de acuerdo a las palabras usadas en 
# el registro de seguimiento de ISOLUCION
patrones = {
    'sin_ejecucion':             'no acredita|sin evidencia|ausencia total',
    'evidencia_incompleta':      'parcial|unicamente|no permite|fase de|solo acredita',
    'control_no_es_control':     'constituye|configura|sentido estricto|gestion operativa',
    'diseño_deficiente':         'periodicidad|desviacion|atributos|no define|no establece',
    'riesgo_mal_formulado':      'causa raiz|causa inmediata|formulacion del riesgo|estructura causal',
    'riesgo_proceso_incorrecto': 'no ha interiorizado|desconexion|dificulta la asignacion|limitacion estructural|su naturaleza corresponde|desalineacion|adscripcion del riesgo|sin el respaldo',
    'tipologia_incorrecta':      'integridad|fraude|tipologia|caracteristicas propias',
    'ejecucion_no_corresponde':  'no corresponde|no da cuenta'
}

In [12]:
# Usando patron-palabra de patrones,recorrer los seguimientos 
# a las evaluaciones de controles, si incluye alguna de las palabras del patron
# se le asigna el patron en el que está esa palabra
# Los patrones usan expresiones regulares donde | significa "o"
# Un control se marca con 1 en una categoría si su texto contiene 
# al menos una de las frases del patrón correspondiente
# El str.contains me da un resultado boleano
for categoria, patron in patrones.items():
    evaluaciones[categoria] = evaluaciones['seguimiento'].str.lower().str.contains(
        patron, na=False
    ).astype(int)

# Crear una variable de nombres de columnas de interés
cols_evaluaciones = ['id_control_iso', 'num', 'eficacia'] + list(patrones.keys())
# Crear la data de problemas con las columnas relevantes de la evaluación
fact_problemas = evaluaciones[cols_evaluaciones].copy()

In [13]:
# La misma mecánica del anterior solo que para las conclusiones de eficacia global
for categoria, patron in patrones.items():
    ef_global[f'{categoria}_global'] = ef_global['eficacia_global_detalle'].str.lower().str.contains(
        patron, na=False
    ).astype(int)

cols_global = ['plan'] + [f'{c}_global' for c in patrones.keys()]

#### Construcción de la data con categorias de principales problemas

In [14]:
# Unir la data de los principales problemas de evaluaciones y eficacia global en una sola
fact_problemas = fact_problemas.merge(
    ef_global[cols_global],
    left_on='num',
    right_on='plan',
    how='left'
).drop(columns='plan')
# El resultado es una columna por categoría con 1 y 0, que representa si aplica o no a cada control.

In [15]:
# Anular dinamización, cada columna de categoría se vuelve un valor de la variable level_3
cols_id = ['id_control_iso', 'num', 'eficacia']
fact_problemas_largo = (
    fact_problemas
    .set_index(cols_id)
    .stack()
    .reset_index()
    .rename(columns={'level_3': 'categoria', 0: 'tiene_problema'})
)
# Eliminar la columna tiene_problema
fact_problemas_largo = fact_problemas_largo[
    fact_problemas_largo['tiene_problema'] == 1
].reset_index(drop=True)

# Crear otra columna de nivel que define si el problema es a nivel de riesgo o de control
fact_problemas_largo['nivel'] = fact_problemas_largo['categoria'].apply(
    lambda x: 'global' if x.endswith('_global') else 'control'
)

fact_problemas_largo['categoria'] = fact_problemas_largo['categoria'].str.replace('_global', '')
fact_problemas_largo = fact_problemas_largo.drop(columns='tiene_problema')

# Exporta el resultado a un archivo plano
exportar_csv(fact_problemas_largo, 'fact_problemas.csv')

## Recalculo del riesgo residual

In [16]:
# cargar datas de los mapas de riesgos
dim_control = pd.read_csv('dim_control.csv', encoding='utf-8-sig', sep='|')
mapa_ini = pd.read_csv('mapa_ini.csv', encoding='utf-8-sig', sep='|')
fact_eficacia = pd.read_csv('fact_eficacia.csv', encoding='utf-8-sig', sep='|')

In [17]:
# Definir pesos para calculos de riesgo residual según guía - Pesos DAFP v7
peso_tipo = {'Preventivo': 0.25, 'Detectivo': 0.15, 'Correctivo': 0.10}
peso_impl = {'Automático': 0.25, 'Manual': 0.15}

In [18]:
controles_eficaces = fact_eficacia[fact_eficacia['eficacia'] == 'SI'].merge(
    dim_control[['id_control', 'tipo_control', 'implementacion_control']],
    left_on='id_control_mapa',
    right_on='id_control',
    how='left'
)

controles_eficaces['peso'] = (
    controles_eficaces['tipo_control'].map(peso_tipo) +
    controles_eficaces['implementacion_control'].map(peso_impl)
)

Calculo de nuevo riesgo residual

In [19]:
# Partir de todos los riesgos del mapa para no excluir 
# los que tienen todos sus controles como no eficaces
resultados = []
for _, fila in mapa_ini.iterrows():
    consecutivo = fila['consecutivo']
    prob_inh = fila['%_probabilidad_inherente']
    imp_inh  = fila['%_impacto_inherente']
    
    # Crear un grupo de los controles eficaces para cada riesgo
    # Si no hay ningun riesgo eficas esto se va en blamnco
    grupo = controles_eficaces[controles_eficaces['consecutivo'] == consecutivo]
    
    # Si no hay riesgos eficaces se deja riesgo residual igual al inherente
    if len(grupo) == 0:
        # Sin controles eficaces: el riesgo residual es igual al inherente
        prob_res, imp_res = round(prob_inh, 4), round(imp_inh, 4)
    # Si hay controles eficaces, se recalcula el riesgo residual
    # Se recalcula porque pueda que hayan controles eficaces y no eficaces para
    # el mismo riesgo
    else:
        prob_res, imp_res = calcular_residual(grupo, prob_inh, imp_inh)
    
    resultados.append({
        'consecutivo': consecutivo,
        'prob_inherente': prob_inh,
        'imp_inherente': imp_inh,
        'prob_residual_calculado': prob_res,
        'imp_residual_calculado': imp_res
    })

df_residual = pd.DataFrame(resultados)
print(df_residual.shape)

(99, 5)


In [20]:
# Traer las etiquetas de probabilidad e impacto residual
df_residual['prob_residual_label'] = df_residual['prob_residual_calculado'].apply(etiqueta_prob)
df_residual['imp_residual_label'] = df_residual['imp_residual_calculado'].apply(etiqueta_imp)
# Traer las etiquetas de riesgo residual final
# Toma cada fila y aplica los calculos
df_residual['zona_residual_calculada'] = df_residual.apply(
    lambda r: zona_riesgo(r['prob_residual_calculado'], r['imp_residual_calculado']), axis=1
)

In [21]:
# Unir todos los datos
df_residual = df_residual.merge(
    mapa_ini[['consecutivo', 'zona_de_riesgo_final', 
              'probabilidad_residual_final', 'impacto_residual_final']],
    on='consecutivo',
    how='left'
)
# Exportar resultados
exportar_csv(df_residual, 'fact_residual.csv')